# 1. Install Library


In [5]:
# !pip install pandas tqdm python-dotenv ollama openpyxl

# 2. Buat File `.env`

In [6]:
import os

os.environ["OLLAMA_API_KEY"] = "ac4a603f12ab485f8003f18441b43006.r5qjZN13Cx7jX8ah1T3FP62H"
os.environ["OLLAMA_MODEL"] = "gemma4:31b-cloud"

In [7]:
print("OLLAMA_MODEL:", os.getenv("OLLAMA_MODEL"))
print("API key ada?:", os.getenv("OLLAMA_API_KEY") is not None)

OLLAMA_MODEL: gemma4:31b-cloud
API key ada?: True


# 3. Siapkan Dataset

# 4. Kode Lengkap

In [ ]:
import re
import json
import time
import csv
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from ollama import Client


# ======================================================
# LOAD ENV
# ======================================================

load_dotenv()


# ======================================================
# KONFIGURASI FILE
# ======================================================

INPUT_FILE = "case_1_labeled_data.xlsx"

OUTPUT_EXCEL = "hasil_label_ollama_cloud.xlsx"
CHECKPOINT_EXCEL = "checkpoint_label_ollama_cloud.xlsx"

TEXT_COLUMN = "full_text"
OLD_LABEL_COLUMN = "label"
NEW_LABEL_COLUMN = "label_baru"

SAVE_EVERY = 10


# ======================================================
# KONFIGURASI OLLAMA CLOUD
# ======================================================

OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "gpt-oss:120b")

if not OLLAMA_API_KEY:
    raise ValueError("OLLAMA_API_KEY belum diisi di file .env")

client = Client(
    host="https://ollama.com",
    headers={
        "Authorization": "Bearer " + OLLAMA_API_KEY
    }
)


# ======================================================
# DAFTAR LABEL
# ======================================================

LABEL_LIST = [
    "Anggaran",
    "Distribusi",
    "Ekonomi",
    "Kualitas Pangan",
    "Lainnya",
    "Politik",
    "Sasaran Penerima",
    "Tata Kelola",
]


# ======================================================
# SYSTEM PROMPT
# ======================================================

SYSTEM_PROMPT = """
Anda adalah annotator data teks berbahasa Indonesia untuk klasifikasi opini publik tentang program MBG atau Makan Bergizi Gratis.

Tugas utama Anda adalah menentukan satu label baru yang paling tepat untuk setiap teks.

Label yang valid hanya boleh salah satu dari daftar berikut:
- Anggaran
- Distribusi
- Ekonomi
- Kualitas Pangan
- Lainnya
- Politik
- Sasaran Penerima
- Tata Kelola

Gunakan label lama sebagai konteks tambahan, tetapi label lama tidak selalu benar.
Prioritaskan isi teks pada kolom full_text.

Definisi label:

1. Anggaran
Gunakan label ini jika teks membahas:
- dana, biaya, anggaran, pemborosan, defisit, subsidi, uang negara
- alokasi dana MBG
- kritik bahwa dana MBG sebaiknya dipakai untuk kebutuhan lain
- pertanyaan seperti "uangnya ke mana", "buang anggaran", "penggarongan duit negara"
Contoh kata/frasa: anggaran, duit negara, dana, biaya, defisit, efisiensi, subsidi, pemborosan, penggarongan, buang uang.

2. Distribusi
Gunakan label ini jika teks membahas:
- penyaluran MBG
- pendistribusian makanan
- keterlambatan program
- daerah yang belum mendapat MBG
- monitoring pembagian makanan
- jumlah siswa/penerima yang menerima makanan dalam kegiatan distribusi
Contoh kata/frasa: distribusi, pendistribusian, penyaluran, monitoring, terlambat, belum jalan, belum ada MBG, menerima makanan, siswa terima.

3. Ekonomi
Gunakan label ini jika teks membahas:
- dampak MBG terhadap ekonomi lokal
- petani, peternak, UMKM, pangan lokal
- kesejahteraan ekonomi masyarakat
- daya beli, harga, pasar, pendapatan
Contoh kata/frasa: ekonomi lokal, petani, peternak, UMKM, pangan lokal, kesejahteraan petani, rakyat sejahtera.

4. Kualitas Pangan
Gunakan label ini jika teks membahas:
- mutu makanan
- menu MBG
- rasa makanan
- makanan tidak disukai
- gizi, makanan sehat, lauk, buah, sayur
- tempat makan, wadah makanan, kebersihan, makanan rusak, keracunan, kelayakan konsumsi
Contoh kata/frasa: menu, makan sehat, gizi, lauk, buah, sayur, tempat makan, makanan basi, keracunan, tidak suka menu.

5. Lainnya
Gunakan label ini hanya jika teks:
- terlalu pendek atau tidak jelas
- hanya candaan, meme, singkatan, atau komentar acak
- menyebut MBG tetapi tidak cukup konteks untuk masuk label lain
- hanya berisi mention, link, atau ujaran tidak informatif

6. Politik
Gunakan label ini jika teks membahas:
- Presiden, menteri, pejabat, partai politik, tokoh politik
- Prabowo, Gerindra, Dasco, Jokowi, Anies, atau aktor politik lain
- kampanye, pencitraan, kebijakan politik, kritik kepada pemerintah
- MBG sebagai agenda politik atau kepentingan politik
Contoh kata/frasa: presiden, menteri, Prabowo, Gerindra, pejabat, politik, kampanye, pencitraan, kebijakan, pemerintahan.

7. Sasaran Penerima
Gunakan label ini jika teks membahas:
- siapa penerima MBG
- anak sekolah, siswa, keluarga miskin, fakir miskin, anak terlantar
- ketepatan penerima
- masyarakat yang merasa diperhatikan
- manfaat MBG bagi anak, murid, atau kelompok target
Contoh kata/frasa: siswa, anak-anak, sekolah, penerima, masyarakat, fakir miskin, anak terlantar, keluarga, rakyat kecil.

8. Tata Kelola
Gunakan label ini jika teks membahas:
- pengawasan program
- mekanisme pelaksanaan
- aturan, administrasi, birokrasi
- transparansi, evaluasi, data, korupsi dalam pelaksanaan
- program perlu diawasi atau diperbaiki sistemnya
Contoh kata/frasa: pengawasan, tata kelola, mekanisme, aturan, transparansi, data, evaluasi, koruptor, pelaksanaan, berjalan lancar, masalah berulang.

Aturan prioritas jika teks memiliki lebih dari satu isu:
- Jika fokus utama adalah uang/dana/biaya, pilih Anggaran.
- Jika fokus utama adalah proses pembagian atau keterlambatan MBG di daerah, pilih Distribusi.
- Jika fokus utama adalah penerima seperti siswa, anak sekolah, fakir miskin, atau masyarakat sasaran, pilih Sasaran Penerima.
- Jika fokus utama adalah menu, mutu, rasa, gizi, makanan, atau wadah makanan, pilih Kualitas Pangan.
- Jika fokus utama adalah presiden, pejabat, partai, atau kritik politik, pilih Politik.
- Jika fokus utama adalah sistem pelaksanaan, pengawasan, data, transparansi, atau evaluasi program, pilih Tata Kelola.
- Jika fokus utama adalah dampak ke petani, UMKM, pangan lokal, pendapatan, atau ekonomi lokal, pilih Ekonomi.
- Jika tidak ada konteks yang cukup, pilih Lainnya.

Contoh klasifikasi:

Teks: "Dari pada mbg perbaikan ruang kelas jauh lbh dibutuhkan"
Output: {"label_baru": "Anggaran"}

Teks: "di Bandung saja MBG terlambat dimulai"
Output: {"label_baru": "Distribusi"}

Teks: "Program MBG melibatkan petani, peternak, dan UMKM sekitar"
Output: {"label_baru": "Ekonomi"}

Teks: "ga semua suka sama menu MBGnya"
Output: {"label_baru": "Kualitas Pangan"}

Teks: "Presiden ngotot, paling sebel kalau MBG dikritik"
Output: {"label_baru": "Politik"}

Teks: "Pasal 34 ayat 1 mengamanatkan negara memelihara fakir miskin dan anak terlantar"
Output: {"label_baru": "Sasaran Penerima"}

Teks: "Lanjutkan MBG dengan pengawasan yang ketat"
Output: {"label_baru": "Tata Kelola"}

Teks: "chaos bareng mbg"
Output: {"label_baru": "Lainnya"}

Jawaban wajib hanya JSON valid.
Tidak boleh ada penjelasan tambahan.
Format wajib:
{"label_baru": "Nama Label"}
"""


# ======================================================
# USER PROMPT
# ======================================================

def build_user_prompt(full_text: str, old_label: str) -> str:
    return f"""
Klasifikasikan teks berikut ke dalam satu label baru.

Label lama:
{old_label}

Teks:
{full_text}

Instruksi:
- Pilih hanya satu label.
- Label harus salah satu dari daftar valid.
- Gunakan label lama hanya sebagai konteks tambahan.
- Keputusan utama berdasarkan isi teks.
- Jawab hanya JSON valid.

Output:
{{"label_baru": "..."}}
""".strip()


# ======================================================
# LOAD DATASET
# ======================================================

def load_dataset(file_path: str) -> pd.DataFrame:
    """
    Load dataset dari Excel atau CSV.

    Format yang didukung:
    - .xlsx
    - .csv
    """

    file_path = str(file_path)
    suffix = Path(file_path).suffix.lower()

    if suffix == ".xlsx":
        df = pd.read_excel(
            file_path,
            engine="openpyxl"
        )

    elif suffix == ".csv":
        encodings = [
            "utf-8-sig",
            "utf-8",
            "cp1252",
            "latin1"
        ]

        df = None
        last_error = None

        for enc in encodings:
            try:
                df = pd.read_csv(
                    file_path,
                    encoding=enc,
                    sep=None,
                    engine="python",
                    quotechar='"',
                    escapechar="\\",
                    on_bad_lines="skip"
                )
                print(f"CSV berhasil dibaca dengan encoding: {enc}")
                break

            except Exception as e:
                last_error = e

        if df is None:
            raise ValueError(f"Gagal membaca CSV. Error terakhir: {last_error}")

    else:
        raise ValueError(
            "Format file tidak didukung. Gunakan file .xlsx atau .csv"
        )

    # Bersihkan nama kolom
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace("\ufeff", "", regex=False)
    )

    required_columns = [TEXT_COLUMN, OLD_LABEL_COLUMN]

    for col in required_columns:
        if col not in df.columns:
            raise ValueError(
                f"Kolom '{col}' tidak ditemukan. "
                f"Kolom yang tersedia: {list(df.columns)}"
            )

    if NEW_LABEL_COLUMN not in df.columns:
        df[NEW_LABEL_COLUMN] = ""

    return df


# ======================================================
# SIMPAN KE EXCEL
# ======================================================

def save_to_excel(df: pd.DataFrame, output_path: str) -> None:
    """
    Simpan DataFrame ke Excel.
    """

    df.to_excel(
        output_path,
        index=False,
        engine="openpyxl"
    )


# ======================================================
# NORMALISASI OUTPUT LLM
# ======================================================

def normalize_label(raw_response: str) -> str:
    """
    Membersihkan output dari LLM.
    Output ideal:
    {"label_baru": "Anggaran"}
    """

    if raw_response is None:
        return "Lainnya"

    raw_response = str(raw_response).strip()

    if not raw_response:
        return "Lainnya"

    cleaned = (
        raw_response
        .replace("```json", "")
        .replace("```JSON", "")
        .replace("```", "")
        .strip()
    )

    # Parse JSON langsung
    try:
        data = json.loads(cleaned)
        label = str(data.get("label_baru", "")).strip()

        if label in LABEL_LIST:
            return label

    except Exception:
        pass

    # Cari JSON di dalam teks jika model menambah teks lain
    json_match = re.search(r"\{.*?\}", cleaned, re.DOTALL)

    if json_match:
        try:
            data = json.loads(json_match.group(0))
            label = str(data.get("label_baru", "")).strip()

            if label in LABEL_LIST:
                return label

        except Exception:
            pass

    # Fallback: cari nama label di response
    for label in LABEL_LIST:
        pattern = r"\b" + re.escape(label) + r"\b"

        if re.search(pattern, cleaned, re.IGNORECASE):
            return label

    return "Lainnya"


# ======================================================
# PANGGIL OLLAMA CLOUD
# ======================================================

def call_ollama_cloud(
    full_text: str,
    old_label: str,
    max_retries: int = 3,
    sleep_seconds: int = 2
) -> str:
    """
    Memanggil Ollama Cloud.
    Response streaming digabung menjadi satu teks penuh.
    """

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": build_user_prompt(full_text, old_label)
        }
    ]

    for attempt in range(max_retries):
        try:
            full_response = ""

            stream = client.chat(
                OLLAMA_MODEL,
                messages=messages,
                stream=True,
                options={
                    "temperature": 0,
                    "num_predict": 100
                }
            )

            for part in stream:
                content = part["message"]["content"]
                full_response += content

            label_baru = normalize_label(full_response)
            return label_baru

        except Exception as e:
            print(f"[ERROR] Percobaan {attempt + 1} gagal: {e}")
            time.sleep(sleep_seconds * (attempt + 1))

    return "Lainnya"


# ======================================================
# PROSES LABELING DATASET
# ======================================================

def labeling_dataset() -> pd.DataFrame:
    df = load_dataset(INPUT_FILE)

    print("Dataset berhasil dimuat")
    print(f"Jumlah data      : {len(df)}")
    print(f"Kolom dataset    : {list(df.columns)}")
    print(f"Model Ollama     : {OLLAMA_MODEL}")
    print(f"Output Excel     : {OUTPUT_EXCEL}")
    print(f"Checkpoint Excel : {CHECKPOINT_EXCEL}")

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Labeling dengan Ollama Cloud"
    ):

        existing_label = str(row.get(NEW_LABEL_COLUMN, "")).strip()

        # Skip jika label_baru sudah ada dan valid
        if existing_label in LABEL_LIST:
            continue

        full_text = row[TEXT_COLUMN]
        old_label = row[OLD_LABEL_COLUMN]

        # Handle teks kosong
        if pd.isna(full_text) or str(full_text).strip() == "":
            df.at[idx, NEW_LABEL_COLUMN] = "Lainnya"
            continue

        # Handle label lama kosong
        if pd.isna(old_label) or str(old_label).strip() == "":
            old_label = "Tidak Ada"

        full_text = str(full_text).strip()
        old_label = str(old_label).strip()

        label_baru = call_ollama_cloud(
            full_text=full_text,
            old_label=old_label
        )

        df.at[idx, NEW_LABEL_COLUMN] = label_baru

        # Simpan checkpoint ke Excel setiap beberapa baris
        if (idx + 1) % SAVE_EVERY == 0:
            save_to_excel(df, CHECKPOINT_EXCEL)
            print(f"\nCheckpoint tersimpan sampai baris ke-{idx + 1}")

    # Simpan hasil akhir ke Excel
    save_to_excel(df, OUTPUT_EXCEL)

    print("\nSelesai.")
    print(f"File hasil akhir disimpan ke: {OUTPUT_EXCEL}")

    return df


# ======================================================
# MAIN
# ======================================================

if __name__ == "__main__":
    hasil_df = labeling_dataset()

    print("\nPreview hasil:")
    print(
        hasil_df[
            [
                TEXT_COLUMN,
                OLD_LABEL_COLUMN,
                NEW_LABEL_COLUMN
            ]
        ].head(10)
    )

c:\SatriaData26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset berhasil dimuat
Jumlah data      : 5000
Kolom dataset    : ['full_text', 'label', 'label_baru']
Model Ollama     : gemma4:31b-cloud
Output Excel     : hasil_label_ollama_cloud.xlsx
Checkpoint Excel : checkpoint_label_ollama_cloud.xlsx


Labeling dengan Ollama Cloud:   0%|          | 10/5000 [00:20<3:15:51,  2.36s/it]


Checkpoint tersimpan sampai baris ke-10


Labeling dengan Ollama Cloud:   0%|          | 20/5000 [00:49<7:51:19,  5.68s/it]

...

Labeling dengan Ollama Cloud:   1%|          | 50/5000 [01:33<1:34:12,  1.14s/it]


Checkpoint tersimpan sampai baris ke-50


Labeling dengan Ollama Cloud:   1%|          | 60/5000 [03:26<24:02:56, 17.53s/it]


Checkpoint tersimpan sampai baris ke-60


Labeling dengan Ollama Cloud:   1%|▏         | 64/5000 [06:46<35:00:04, 25.53s/it]

---

# 5. Jalankan Script
